<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/01_chunk_documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
!git clone https://github.com/yaranoun/ML-Tech.git

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: could not create work tree dir 'ML-Tech': No such file or directory


In [68]:
from pathlib import Path

%cd ML-Tech
raw_folder = Path("data/raw")

documents = []

for file_path in raw_folder.glob("*.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append({
        "filename": file_path.name,
        "content": text
    })

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(doc["filename"])

[Errno 2] No such file or directory: 'ML-Tech'
/content/ML-Tech/ML-Tech
Loaded 0 documents


In [69]:
document_sections = {
    "Biometric Passport.txt": [
        "Requested documents:",
        "Remarks:",
        "Fees:",
        "NB:"
    ],

    "Lost Passport or Stolen Passport.txt": [
        "Lost Passport:",
        "Stolen Passport:",
        "NB:"
    ],

    "Ex-porting Biometric Passport.txt": [
        "Exporting a Lebanese passport",
        "Exporting a Foreign passport",
        "Lebanese or foreign passport shipped:",
        "For travel agencies that plan to ship passports:",
        "For the individual planning on shipping his passport with another traveler:",
        "Nb:"
    ],
    "Passport of adopted, born in special circumstances child or a citizen without a family name.txt":[
        "The requested documents",
        "Child born in special cirmustances",
        "A passport for a minor:",
        "A citizen without a family name"
    ],
    "Personal attendance required.txt":[
        "Personal attendance required",
        "Exemption from attendance",
        "Exemption from fees"
    ],
    "Certifying Passport.txt":[]
}

In [70]:
def extract_metadata(text):
    metadata = {
        "url": "",
        "title": "",
        "category": "",
        "keywords": ""
    }

    # Separate metadata from actual content
    if "Content:" in text:
        metadata_part, content = text.split("Content:", 1)
    else:
        metadata_part = ""
        content = text

    # Extract each metadata field
    for line in metadata_part.splitlines():
        line = line.strip()

        if line.startswith("URL:"):
            metadata["url"] = line.replace("URL:", "", 1).strip()

        elif line.startswith("Title:"):
            metadata["title"] = line.replace("Title:", "", 1).strip()

        elif line.startswith("Category:"):
            metadata["category"] = line.replace("Category:", "", 1).strip()

        elif line.startswith("Keywords:"):
            metadata["keywords"] = line.replace("Keywords:", "", 1).strip()

    return metadata, content.strip()

In [71]:
metadata, content = extract_metadata(documents[0]["content"])

print(metadata)
print(content)

IndexError: list index out of range

In [ ]:
import re

def chunk_document(filename, text, headings,metadata):
    chunks = []

    # If there are no headings, keep the whole document
    if not headings:
        chunks.append({
            "document": filename,
            "title":metadata["title"],
            "url":metadata["url"],
            "category":metadata["category"],
            "keywords":metadata["keywords"],
            "section": "Full Document",
            "text": text.strip()
        })
        return chunks

    # Build a regex from the headings
    pattern = "|".join(re.escape(h) for h in headings)

    # Split while keeping the headings
    parts = re.split(f"({pattern})", text)

    current_heading = None
    current_text = ""

    for part in parts:

        if part in headings:

            if current_heading is not None:
                chunks.append({
                    "document": filename,
                     "title":metadata["title"],
                    "url":metadata["url"],
                    "category":metadata["category"],
                    "keywords":metadata["keywords"],
                    "section": current_heading,
                    "text": current_text.strip()
                })

            current_heading = part.rstrip(":")
            current_text = ""

        else:
            current_text += part

    # Save the last chunk
    if current_heading is not None:
        chunks.append({
            "document": filename,
            "title":metadata["title"],
            "url":metadata["url"],
            "category":metadata["category"],
            "keywords":metadata["keywords"],
            "section": current_heading,
            "text": current_text.strip()

        })

    return chunks

In [ ]:
all_chunks = []
for document in documents:
    filename = document["filename"]
    raw_text = document["content"]

    # remove URL, title, category, keywords, etc.
    metadata,clean_text = extract_metadata(raw_text)

    headings = document_sections.get(filename, [])

    chunks = chunk_document(
        filename,
        clean_text,
        headings,
        metadata
    )

    all_chunks.extend(chunks)

In [ ]:
lost_chunk = None
stolen_chunk = None

for chunk in all_chunks:
    if chunk["section"] == "Lost Passport":
        lost_chunk = chunk

    elif chunk["section"] == "Stolen Passport":
        stolen_chunk = chunk

In [ ]:
if lost_chunk and stolen_chunk:

    stolen_chunk["text"] = (
        "Lost passport procedure:\n"
        + lost_chunk["text"]
        + "\n\n"
        + "Additional information for a stolen passport:\n"
        + stolen_chunk["text"]
    )

In [ ]:
import json
import os

os.makedirs("data/processed", exist_ok=True)

with open("data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=4, ensure_ascii=False)

In [ ]:
import json

with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(json.dumps(chunks, indent=4, ensure_ascii=False))

In [ ]:
!git config --global user.email "yarajnoun@gmail.com"
!git config --global user.name "yaranoun"

In [72]:
!git add data/processed/chunks.json
!git commit -m "Update document chunks"
!git pull --rebase origin main
!git push origin main

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory


In [73]:
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/chunks.json'

In [ ]:
!ls

In [ ]:
!rm -rf /content/ML-Tech/ML-Tech